In [2]:
!pip install -q -U langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 16.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


In [3]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")
MODEL="gemini-3.5-flash-lite"


In [11]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, ToolMessage

llm = ChatGoogleGenerativeAI(model=MODEL)
expenses = []

@tool
def add_expense(category: str, amt: float) -> float:
  """Tool to add expense category and amount to expense list"""
  expenses.append({"category": category, "amount": amt})
  return f"CATEGORY:{category} added to expense with amount:{amt}"

@tool
def get_total_expense() -> float:
  """Tool to calculate total expense"""
  total = 0
  for expense in expenses:
    total += expense["amount"]
  return total

@tool
def get_category_expense(category: str) -> float:
  """Tool to calculate particular category expense"""
  total = 0
  for expense in expenses:
    if expense["category"].lower() == category.lower():
      total += expense["amount"]
  return total

@tool
def get_expense() -> list:
  """Tool to get all expenses"""
  return expenses

llm_with_tools = llm.bind_tools([add_expense,get_total_expense,get_category_expense,get_expense])

def chat(query):
  messages = []
  messages.append(HumanMessage(content=query))
  AI = llm_with_tools.invoke(messages)
  messages.append(AI)
  if not AI.tool_calls:
    return AI.content[0]["text"]
  tool_call=AI.tool_calls[0]
  tools={
      "add_expense":add_expense,
      "get_total_expense":get_total_expense,
      "get_category_expense":get_category_expense,
      "get_expense":get_expense
  }

  selected_tool=tools[tool_call["name"]]
  tool_result=selected_tool.invoke(tool_call["args"])
  tool_message=ToolMessage(content=str(tool_result),name=tool_call["name"],tool_call_id=tool_call["id"])
  messages.append(tool_message)
  final_output = llm_with_tools.invoke(messages)
  return final_output.content[0]["text"]


print("Enter bye/exit/quit to terminate session")
while True:
  user = input("Enter your query: ").strip()
  if user.lower() in ['exit', 'bye', 'quit']:
    print("exited")
    break
  output = chat(user)
  print(output)

Enter bye/exit/quit to terminate session
Enter your query: i ate birayni for 129rs
I've added 129 Rs for food to your expenses.
Enter your query: my food expenses
Your total food expenses are $129.
Enter your query: not dollars its in ruppes
Could you please tell me which expense you would like to add? For example, you can say something like "Add 500 rupees for food."
Enter your query: i took books for 100rs
I've added 100rs for books to your expenses.
Enter your query: tell me study expense
Your study expense is 0.
Enter your query: tell me by book expense
Your book expense is 0.
Enter your query: tell my books expense
Your total expense for books is 100.
Enter your query: ok my total expense
Your total expense is 229.
Enter your query: tell me my expenses
Your expenses are:
- Food: 129.0
- Books: 100.0
Enter your query: i ate roti for 60rs
I have added Rs. 60 for "roti" to your expenses.
Enter your query: tell my expenses
Your expenses are:
- Food: 129.0
- Books: 100.0
- Roti: 60.0
E